# Exercise 10 — Charging Station Placement


## Task

A bus operator wants to electrify a route with 10 stations. Charging stations cost **EUR 180,000 each**, so their number should be minimized subject to battery feasibility over **10 complete rounds**.

Parameters:

- battery capacity: **60 kWh**;
- charging rate: **3 kWh/min**;
- electricity consumption: **2.6 kWh/km**;
- minimum safety buffer: **30% of battery capacity**;
- ordinary stop time: **60 s = 1 min**;
- stop time at stations 5 and 10: **180 s = 3 min**;
- the bus can start with a full battery at station 1.

### Route distances

The line follows the cyclic route

`1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9 -> 10 -> 1`.

| Arc | Distance |
|---|---:|
| 1 -> 2 | 0.350 km |
| 2 -> 3 | 0.350 km |
| 3 -> 4 | 0.500 km |
| 4 -> 5 | 0.400 km |
| 5 -> 6 | 0 km |
| 6 -> 7 | 0.400 km |
| 7 -> 8 | 0.500 km |
| 8 -> 9 | 0.350 km |
| 9 -> 10 | 0.350 km |
| 10 -> 1 | 0 km |

The zero-length transitions correspond to the same physical station, but opposite side of the street. 

Tasks:

1. Formulate the charging-station placement problem algebraically.
2. Translate it into JuMP and solve it.
3. Interpret which stations receive chargers and verify the battery trajectory.

## Reference implementation


In [1]:
using JuMP, HiGHS

S = 1:10
d = Dict(
    (1, 2) => 0.350,
    (2, 3) => 0.350,
    (3, 4) => 0.500,
    (4, 5) => 0.400,
    (5, 6) => 0.0,
    (6, 7) => 0.400,
    (7, 8) => 0.500,
    (8, 9) => 0.350,
    (9, 10) => 0.350,
    (10, 1) => 0.0,
)

w = Dict(s => (s in (5, 10) ? 3 : 1) for s in S)

f = 3

cap = 60

c = 2.6

R = 1:10

function prev_fun(r, s, R, S)
    if s!=1
        return (r, s-1)
    elseif r==1 && s==1
        return (R[end], S[end])
    else
        return r-1, S[end]
    end
end

rs_prev = Dict((r, s) => prev_fun(r, s, R, S) for r in R, s in S)
s_prev = Dict(s => (s==1 ? S[end] : s-1) for s in S)

cap = 60

m = Model(HiGHS.Optimizer)

@variable(m, x[S], Bin)
@variable(m, 0.3*cap <= ch[R, S] <= cap)

@objective(m, Min, 180_000 * sum(x[s] for s in S) + 1e-9 * sum(ch[r, s] for r in R, s in S))
@constraint(
    m,
    balance[r in R, s in S],
    ch[r, s] <= ch[rs_prev[r, s]...] + x[s] * f * w[s] - d[s_prev[s], s]*c
)

optimize!(m)

value.(x)
value.(ch)


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
MIP has 100 rows; 110 cols; 300 nonzeros; 10 integer variables (10 binary)
Coefficient ranges:
  Matrix  [1e+00, 9e+00]
  Cost    [1e-09, 2e+05]
  Bound   [1e+00, 6e+01]
  RHS     [9e-01, 1e+00]
Presolving model
100 rows, 110 cols, 300 nonzeros 0s
100 rows, 110 cols, 300 nonzeros 0s
Presolve reductions: rows 100(-0); columns 110(-0); nonzeros 300(-0) - Not reduced

Solving MIP model with:
   100 rows
   110 cols (10 binary, 0 integer, 0 implied int., 100 continuous, 0 domain fixed)
   300 nonzeros
   Thread count 5 (of 10 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate 

2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:10
    Dimension 2, 1:10
And data, a 10×10 Matrix{Float64}:
 21.4441  20.5341  19.6241  18.3241  …  24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241  …  24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441
 21.4441  20.5341  19.6241  18.3241     24.5641  23.2641  22.3541  21.4441